# Kalibrierung der Datenparameter für 4 binäre Kovariaten

Kovariaten:
- X_1 ~ Bernoulli(0.3), X_2 ~ Bernoulli(0.8)  [unverändert]
- X_3 ~ Bernoulli(0.9), X_4 ~ Bernoulli(0.4)  [neu: binär statt kontinuierlich]

Koeffizienten:
- p_1=-0.405, p_2=-0.4, p_3=-0.35, p_4=-0.30

Zwei Kalibrierungsziele:
- **events20**: ~20% Events vor τ, ~10%/30%/50% Zero-Weights
- **events40**: ~40% Events vor τ, ~10%/30%/50% Zero-Weights

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import create_weibull_data, create_data_with_ipc_weights, simulation

seed = 42
n = 1429
tau = 37

X_pred_point = pd.DataFrame({'X_1': [0], 'X_2': [1], 'X_3': [1], 'X_4': [0]})

base_params = {
    'shape_weibull': 1,
    'p_1': -0.405,
    'p_2': -0.4,
    'p_3': -0.35,
    'p_4': -0.30,
    'n': n,
    'tau': tau,
    'X_pred_point': X_pred_point,
    'covariate_type': 'binary',
    'scale_weibull_base': 400,
    'rate_censoring': 0.003,
}

params_rf = {
    'n_estimators': 50,
    'max_depth': 4,
    'min_samples_split': 5,
    'max_features': 'log2',
    'random_state': seed,
    'weighted_bootstrapping': True,
}

# Überprüfe Datenerzeugung
test_data = create_weibull_data(base_params, seed)
print('Datentypen:', test_data.dtypes)
print('Erste Zeilen:')
test_data.head()

Datentypen: X_1        int64
X_2        int64
X_3        int64
X_4        int64
time     float64
event      int64
dtype: object
Erste Zeilen:


,X_1,X_2,X_3,X_4,time,event
0,1,1,1,0,278.612007,0
1,0,0,1,1,12.967089,1
2,1,1,1,1,25.400585,1
3,0,1,1,0,48.329318,1
4,0,1,1,0,0.357090,0


In [32]:
# Überprüfe Verteilung der Survived-Spalte mit aktuellen Parametern
bb = create_data_with_ipc_weights(base_params, test_data.copy())
print('Survived-Verteilung (survived=0: Event, survived=1: kein Event bis tau, survived=999: zensiert vor tau):')
print(bb['survived'].value_counts(normalize=True).round(4))

Survived-Verteilung (survived=0: Event, survived=1: kein Event bis tau, survived=999: zensiert vor tau):
survived
1      0.7131
0      0.1868
999    0.1001
Name: proportion, dtype: float64


In [33]:
def find_parameters(target_event_portion, target_censored_portion,
                    initial_scale, initial_rate,
                    base_params, seed=42):
    """Iterative Kalibrierung von scale_weibull_base und rate_censoring.

    Angepasst an covariate_type='binary' (X_3~B(0.9), X_4~B(0.4)).
    """
    params = dict(base_params)
    params['scale_weibull_base'] = initial_scale
    params['rate_censoring'] = initial_rate

    tolerance = 0.001
    max_iterations = 1000
    ss = 0.01

    for iteration in range(max_iterations):
        result = simulation(
            seed=seed,
            data_generation_parameter=params,
            params_rf={'n_estimators': 10, 'max_depth': 4, 'min_samples_split': 5,
                       'max_features': 'log2', 'random_state': seed,
                       'weighted_bootstrapping': True},
            train_models=False,
            ijk_wager_calc=False,
            ijk_butt_calc=False,
            ijk_jahn_calc=False,
            jk_wager_calc=False,
            boot_calc=[False, 0],
        )

        survived_train = result[9]
        event_portion = survived_train.get(0, 0.0)
        censored_portion = survived_train.get(999, 0.0)

        if (abs(event_portion - target_event_portion) < tolerance and
                abs(censored_portion - target_censored_portion) < tolerance):
            print(f'Ziel erreicht nach {iteration+1} Iterationen:')
            print(f'  scale_weibull_base = {params["scale_weibull_base"]:.4f}')
            print(f'  rate_censoring     = {params["rate_censoring"]:.6f}')
            print(f'  Anteile: Events={event_portion:.4f}, Zensiert={censored_portion:.4f}')
            break

        if event_portion < target_event_portion:
            params['scale_weibull_base'] -= params['scale_weibull_base'] * ss
        else:
            params['scale_weibull_base'] += params['scale_weibull_base'] * ss

        if censored_portion < target_censored_portion:
            params['rate_censoring'] += params['rate_censoring'] * ss
        else:
            params['rate_censoring'] -= params['rate_censoring'] * ss
    else:
        print(f'Warnung: Max. Iterationen ({max_iterations}) erreicht.')
        print(f'  scale={params["scale_weibull_base"]:.4f}, rate={params["rate_censoring"]:.6f}')
        print(f'  Erreichte Anteile: Events={event_portion:.4f}, Zensiert={censored_portion:.4f}')

    return params['scale_weibull_base'], params['rate_censoring']

print('Kalibrierungsfunktion definiert.')

Kalibrierungsfunktion definiert.


## Kalibrierung: events20 (~20% Events vor τ)

Ziele:
- Szenario 1: ~10% Zero-Weights, ~20% Events
- Szenario 3: ~30% Zero-Weights, ~20% Events
- Szenario 5: ~50% Zero-Weights, ~20% Events

In [34]:
print('=== events20: Szenario 1 (10% Zero-Weights) ===')
scale_20_1, rate_20_1 = find_parameters(
    target_event_portion=0.20,
    target_censored_portion=0.10,
    initial_scale=400,
    initial_rate=0.003,
    base_params=base_params,
)
print()

=== events20: Szenario 1 (10% Zero-Weights) ===
Ziel erreicht nach 214 Iterationen:
  scale_weibull_base = 369.0055
  rate_censoring     = 0.003059
  Anteile: Events=0.2000, Zensiert=0.1000



In [35]:
print('=== events20: Szenario 3 (30% Zero-Weights) ===')
scale_20_3, rate_20_3 = find_parameters(
    target_event_portion=0.20,
    target_censored_portion=0.30,
    initial_scale=350,
    initial_rate=0.015,
    base_params=base_params,
)
print()

=== events20: Szenario 3 (30% Zero-Weights) ===
Warnung: Max. Iterationen (1000) erreicht.
  scale=332.9295, rate=0.011002
  Erreichte Anteile: Events=0.1970, Zensiert=0.2980



In [36]:
print('=== events20: Szenario 5 (50% Zero-Weights) ===')
scale_20_5, rate_20_5 = find_parameters(
    target_event_portion=0.20,
    target_censored_portion=0.50,
    initial_scale=300,
    initial_rate=0.030,
    base_params=base_params,
)
print()

=== events20: Szenario 5 (50% Zero-Weights) ===
Ziel erreicht nach 65 Iterationen:
  scale_weibull_base = 281.6261
  rate_censoring     = 0.022153
  Anteile: Events=0.2000, Zensiert=0.5000



In [37]:
print('=== Ergebnisse events20 ===')
print(f'Szenario 1:  scale_weibull_base={scale_20_1:.2f},  rate_censoring={rate_20_1:.6f}')
print(f'Szenario 3:  scale_weibull_base={scale_20_3:.2f},  rate_censoring={rate_20_3:.6f}')
print(f'Szenario 5:  scale_weibull_base={scale_20_5:.2f},  rate_censoring={rate_20_5:.6f}')

=== Ergebnisse events20 ===
Szenario 1:  scale_weibull_base=369.01,  rate_censoring=0.003059
Szenario 3:  scale_weibull_base=332.93,  rate_censoring=0.011002
Szenario 5:  scale_weibull_base=281.63,  rate_censoring=0.022153


## Kalibrierung: events40 (~40% Events vor τ)

Ziele:
- Szenario 1: ~10% Zero-Weights, ~40% Events
- Szenario 3: ~30% Zero-Weights, ~40% Events
- Szenario 5: ~50% Zero-Weights, ~40% Events

In [38]:
print('=== events40: Szenario 1 (10% Zero-Weights) ===')
scale_40_1, rate_40_1 = find_parameters(
    target_event_portion=0.40,
    target_censored_portion=0.10,
    initial_scale=80,
    initial_rate=0.003,
    base_params=base_params,
)
print()

=== events40: Szenario 1 (10% Zero-Weights) ===
Ziel erreicht nach 246 Iterationen:
  scale_weibull_base = 167.3021
  rate_censoring     = 0.003308
  Anteile: Events=0.4000, Zensiert=0.1000



In [39]:
print('=== events40: Szenario 3 (30% Zero-Weights) ===')
scale_40_3, rate_40_3 = find_parameters(
    target_event_portion=0.40,
    target_censored_portion=0.30,
    initial_scale=70,
    initial_rate=0.015,
    base_params=base_params,
)
print()

=== events40: Szenario 3 (30% Zero-Weights) ===
Warnung: Max. Iterationen (1000) erreicht.
  scale=131.4354, rate=0.013171
  Erreichte Anteile: Events=0.3920, Zensiert=0.3020



In [40]:
print('=== events40: Szenario 5 (50% Zero-Weights) ===')
scale_40_5, rate_40_5 = find_parameters(
    target_event_portion=0.40,
    target_censored_portion=0.50,
    initial_scale=55,
    initial_rate=0.030,
    base_params=base_params,
)
print()

=== events40: Szenario 5 (50% Zero-Weights) ===
Warnung: Max. Iterationen (1000) erreicht.
  scale=86.2584, rate=0.032175
  Erreichte Anteile: Events=0.3940, Zensiert=0.5020



In [41]:
print('=== Ergebnisse events40 ===')
print(f'Szenario 1:  scale_weibull_base={scale_40_1:.2f},  rate_censoring={rate_40_1:.6f}')
print(f'Szenario 3:  scale_weibull_base={scale_40_3:.2f},  rate_censoring={rate_40_3:.6f}')
print(f'Szenario 5:  scale_weibull_base={scale_40_5:.2f},  rate_censoring={rate_40_5:.6f}')

=== Ergebnisse events40 ===
Szenario 1:  scale_weibull_base=167.30,  rate_censoring=0.003308
Szenario 3:  scale_weibull_base=131.44,  rate_censoring=0.013171
Szenario 5:  scale_weibull_base=86.26,  rate_censoring=0.032175


In [42]:
print('\n=== Zusammenfassung: Parameter für run_full_pipeline_4binary.py ===')
print()
print('# events20')
print(f'# Szenario 1:  scale_weibull_base={scale_20_1:.2f},  rate_censoring={rate_20_1:.6f}')
print(f'# Szenario 3:  scale_weibull_base={scale_20_3:.2f},  rate_censoring={rate_20_3:.6f}')
print(f'# Szenario 5:  scale_weibull_base={scale_20_5:.2f},  rate_censoring={rate_20_5:.6f}')
print() 
print('# events40')
print(f'# Szenario 1:  scale_weibull_base={scale_40_1:.2f},  rate_censoring={rate_40_1:.6f}')
print(f'# Szenario 3:  scale_weibull_base={scale_40_3:.2f},  rate_censoring={rate_40_3:.6f}')
print(f'# Szenario 5:  scale_weibull_base={scale_40_5:.2f},  rate_censoring={rate_40_5:.6f}')


=== Zusammenfassung: Parameter für run_full_pipeline_4binary.py ===

# events20
# Szenario 1:  scale_weibull_base=369.01,  rate_censoring=0.003059
# Szenario 3:  scale_weibull_base=332.93,  rate_censoring=0.011002
# Szenario 5:  scale_weibull_base=281.63,  rate_censoring=0.022153

# events40
# Szenario 1:  scale_weibull_base=167.30,  rate_censoring=0.003308
# Szenario 3:  scale_weibull_base=131.44,  rate_censoring=0.013171
# Szenario 5:  scale_weibull_base=86.26,  rate_censoring=0.032175
